# 🎯 [K12_M09] 신규 전입자용 문답 만들기
### 실습 솔루션 & 정답 해설 (Google Colab / Jupyter)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/K12/k12_m09_solution.ipynb)

---

## 📌 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 / 과제** | 일반행정·부대관리 (Level 1) / K12 행정 업무 FAQ 챗봇 설계 |
| **미션** | 미션 09. 신규 전입자용 문답 만들기 (소요시간: 약 1.5시간) |
| **실습 유형** | 생성형 AI 시스템 프롬프트 엔지니어링 및 팩트체크 검증 워크북 |

> **⚠️ [교육용 가상 자료 원칙]**
> 본 실습의 모든 자료는 안전 규정에 따라 **교육용 가상 부대 자료(가상 한빛대대)**로 구성되어 있습니다.
> 실제 군사 기밀이나 부대 세부 인적·작전 정보는 외부 AI 도구에 절대 입력하지 않습니다.



In [ ]:
# ============================================================================
# [환경 설정 및 헬퍼 함수] 마크다운 시각화 렌더러 및 가상 데이터셋 로더
# ============================================================================
from IPython.display import display, Markdown
import json
import os

def load_dataset(mission_num):
    """미션 번호(1~10)에 해당하는 가상 데이터셋 JSON을 로드하여 반환합니다."""
    filename = f"k12_m{mission_num:02d}_dataset.json"
    for base in [".", "uploads/K12", "../uploads/K12"]:
        filepath = os.path.join(base, filename)
        if os.path.exists(filepath):
            with open(filepath, "r", encoding="utf-8") as f:
                return json.load(f)
    print(f"⚠️ 데이터셋 파일을 찾을 수 없습니다: {filename}")
    return None

def show_dataset_summary(mission_num):
    """해당 미션의 데이터셋 주요 항목을 요약 출력합니다."""
    data = load_dataset(mission_num)
    if data:
        display(Markdown(f"**[데이터셋 로드 성공]** `{data.get('title', '')}` ({data.get('mission', '')})"))
        keys = [k for k in data.keys() if k not in ["project", "mission", "title"]]
        print(f"포함 항목: {', '.join(keys)}")

def show_result(mission_title, prompt, result):
    """학습자가 입력한 프롬프트와 AI 답변 및 검증 결과를 시각적으로 렌더링하는 헬퍼 함수"""
    p_text = prompt.strip() if prompt.strip() else "(작성된 프롬프트가 없습니다)"
    r_text = result.strip() if result.strip() else "(기록된 결과가 없습니다)"
    
    md_content = f"""
---
### 📌 [{mission_title}] 실습 기록 리포트

#### 💬 입력 프롬프트
```text
{p_text}
```

#### 📋 AI 생성 결과 및 팩트체크 검증 내역
{r_text}
---
"""
    display(Markdown(md_content))

print("✅ 워크북 시각화 헬퍼 함수 및 가상 데이터셋 로더(k12_m01~m10) 준비 완료!")



---

## 🎯 미션 09. 신규 전입자용 문답 만들기
**소요시간**: 약 1.5시간

![미션 9 신규 전입 신병 첫 주 온보딩 행정 타임라인 및 군 행정 용어 순화 대조표 다이어그램](/uploads/common/K12_m09_onboarding_timeline.png)

### 📋 미션 개요 및 배경
구축된 행정 FAQ 챗봇 시스템을 활용하여 낯선 군 환경에 적응해야 하는 신규 전입 장병 맞춤형 온보딩 행정 가이드를 제작합니다. 전입 첫 주 동안 필수적으로 완료해야 할 행정 절차(서류 제출, 계정 생성, 피복 수령 등)를 우선순위와 시계열 순서로 재배치하고, 이해하기 어려운 군사 행정 전문 용어를 알기 쉬운 표준어로 순화합니다. 특정 사용자 집단의 눈높이에 맞춘 타깃형 행정 지원 콘텐츠 제작 및 프롬프트 응용 역량을 완성합니다.

### 🎯 미션 학습목표
전입 첫날부터 첫 주까지 처리할 행정 절차(전입 서류, 계정, 피복, 휴가 신청 방법 등)를 FAQ에서 골라 순서대로 안내하는 문답 5개를 챗봇으로 만듭니다. 신규 전입자가 이해하기 어려운 용어는 쉬운 말로 바꿉니다. **이를 통해 구축된 챗봇 지식을 기반으로 특정 사용자(신규 전입 장병)의 눈높이에 맞춰 군사 행정 용어를 순화하고 시계열 가이드를 제작하는 온보딩 콘텐츠 응용 역량을 익힙니다.**

### 📑 실습 가이드 및 세부 단계 (프롬프트 & 검증 포인트)
#### 🔹 Step 1. 전입 첫 주에 필요한 절차를 FAQ에서 고른다.
#### 이 단계의 역할

전입 첫 주 동안 신병이 반드시 처리해야 하는 필수 행정 과업(전입 서류 제출, 인트라넷 계정 생성, 피복/장구류 교환, 첫 휴가 신청 절차 등)을 FAQ에서 선별합니다.

#### 해야 할 일

- 20개 행정 FAQ 중 신규 전입 장병이 7일 이내에 반드시 숙지·이행해야 할 과업을 탐색합니다.

- 전입 행정 처리의 시급성과 중요도를 기준으로 5개 핵심 행정 과업을 선정합니다.

- AI에게 신병 행정 온보딩 관점에서 선별된 항목의 적합성을 평가받습니다.

#### 💡 AI 프롬프트 질의 예시

> **[신규 전입 장병 필수 행정 과업 추출 질의]:** "전체 행정 FAQ 20건 중 새로 자대에 전입 온 신병이 첫 주(D+1 ~ D+7)에 반드시 이행해야 하거나 가장 필요로 하는 핵심 행정 절차 5가지를 선별하고 그 이유를 설명해 줘."

#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **적시성(Timeliness):** 첫 주 적응에 실질적으로 필요한 절차가 골고루 선정되었는가?
- **과업 타당성:** 행정 서류, 전산 계정, 생활 물품 등 균형 있게 구성되었는가?

#### 핵심 확인 지표

> 전입 첫 주 5대 필수 행정 절차 선별표

#### 학습자가 제출해야 하는 내용

- 전입자 우선 행정 절차 목록

#### 🔹 Step 2. 순서대로 안내하는 문답 5개를 만든다.
#### 이 단계의 역할

선별된 5개 과업을 전입 당일(D-day)부터 3일 차, 7일 차의 시계열 순서로 재배열하고 챗봇 질의응답 시나리오 5세트를 구성합니다.

#### 해야 할 일

- 선별된 5개 절차를 [D-day 전입 등록 → D+1 인트라넷 계정 → D+3 피복 점검 → D+5 증명서/지원금 → D+7 첫 휴가 절차] 시계열로 정렬합니다.

- 신병이 실제로 물어볼 만한 자연스러운 질문과 챗봇의 단계별 답변으로 구성된 문답 세트 5개를 작성합니다.

- 각 답변의 내용이 FAQ 원문과 일치하는지 대조합니다.

#### 💡 AI 프롬프트 질의 예시

> **[전입 첫 주 시계열 행정 문답 생성]:** "선별된 5개 행정 절차를 신병의 시간 흐름[전입 당일 → 3일 차 → 1주일 차] 순으로 배열하고, 신병이 질문하고 챗봇이 명쾌하게 단계별로 답하는 문답 세트 5개를 작성해 줘."

#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **시간 순서 적절성:** 행정 처리의 논리적 선후 관계가 시계열에 부합하는가?
- **데이터 일치:** 안내 문답 속 구비 서류와 제출처가 FAQ 원문과 정확히 일치하는가?

#### 핵심 확인 지표

> 전입 첫 주 시계열 행정 질의응답 5선

#### 학습자가 제출해야 하는 내용

- 전입자용 순서별 행정 문답 세트

#### 🔹 Step 3. 어려운 용어를 쉬운 말로 바꾼다.
#### 이 단계의 역할

낯선 군사 행정 약어 및 행정 용어(인편, 공람, 기안, 조신 등)를 사회 초년생도 즉각 이해할 수 있는 알기 쉬운 쉬운 우리말로 순화합니다.

#### 해야 할 일

- 5개 문답에서 군 특유의 딱딱한 관공서 표현이나 군사 행정 용어를 발굴합니다.

- AI에게 의미를 정확히 유지하면서 신병 눈높이에 맞춘 친절한 설명어로 윤문하도록 요청합니다.

- 용어 순화 전후 대조표(예: '상신' → '승인 요청', '인편 제출' → '직접 서류 전달')를 작성합니다.

#### 💡 AI 프롬프트 질의 예시

> **[군 행정 용어 쉬운 말 순화 질의]:** "작성된 신병 행정 문답에서 군 특유의 딱딱하고 어려운 행정 용어나 축약어를 찾아내고, 신병 눈높이에 맞춘 친절하고 명확한 표준어로 순화해 줘. (예: 결재 상신 → 지휘관 승인 요청)"

#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **원 의미 보존:** 쉽게 바꾸는 과정에서 행정 규정의 본래 조건이나 서식명이 왜곡되지 않았는가?
- **친절도:** 초임 장병이 위압감 없이 편안하게 절차를 이해할 수 있는 어투인가?

#### 핵심 확인 지표

> 군 행정 용어 순화 대조표 및 순화된 최종 문답 5종

#### 학습자가 제출해야 하는 내용

- 용어 순화 전후 비교표 및 최종 문답

#### 🔹 Step 4. 심화 도전 과제 및 최적화
#### 이 단계의 역할

전입 신병이 주말 동안 가족이나 지인에게 행정 처리 결과를 알릴 수 있도록 '첫 주 행정 체크리스트 카드'를 추가 제작합니다.

#### 해야 할 일

- 5개 문답의 핵심 행동을 [ ] 체크박스 형태의 초간단 점검 카드로 요약합니다.

- 스마트폰 화면 1장에 쏙 들어오는 가독성 높은 레이아웃을 구성합니다.

#### 💡 AI 프롬프트 질의 예시

> **[모바일용 첫 주 행정 체크리스트 생성]:** "전입 장병이 첫 주 행정 처리를 완료했는지 스스로 체크할 수 있는 [체크리스트 카드 요약본]을 마크다운으로 깔끔하게 만들어 줘."

#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **간결성:** 한눈에 완료 여부를 체크할 수 있도록 핵심만 압축되었는가?

#### 핵심 확인 지표

> 신병 첫 주 행정 점검 체크리스트 카드

#### 학습자가 제출해야 하는 내용

- 심화 도전 과제 메모

### 📦 미션 필수 학습자료 및 팁
**미션 수행 필수 학습자료**

- **전입 신병 온보딩 행정 타임라인 시트 (Day 1 ~ Day 7):** 전입 당일, 3일 차, 첫 주말 시계열 행정 처리 순서 템플릿
- **1-Page 신규 전입자용 필수 행정 가이드 서식 (MD):** A4 1장 규격의 온보딩 질의응답 5선 및 행정 체크리스트 마크다운 레이아웃
- **[학습 가이드] 군 행정 용어 순화 사전 및 타깃 맞춤형 콘텐츠 기획 가이드:** 신병 눈높이에 맞춘 행정 용어 순화(상신→승인요청 등) 및 친절한 대화형 시나리오 설계법

**실습 팁:** '결재 상신', '인편 제출' 같은 군사 행정 용어는 신병에게 생소하므로, 원래의 규정 의미를 살리면서도 알기 쉬운 일상 표준어로 풀어서 설명해 주세요.

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [선택] 대상 맞춤 활용 — 신규 전입 장병의 첫 주 적응에 필수적인 행정 과업을 시계열로 구성하고, 어려운 군 행정 용어를 쉬운 말로 순화했는가? (PASS / FAIL)

#### 미션 PASS 조건

- 전입 첫 주(D-day ~ D+7) 시계열 행정 질의응답 5선 완성

- 행정 사실 정보(서류, 기한, 부서)의 FAQ 원문 일치성 100%

- 군 행정 전문 용어 순화 대조표 및 순화 반영도 충족

#### 채점자 판정 메모

> '결재 상신'이나 '인편 제출' 같은 용어가 '승인 요청', '직접 전달' 등 초임자 눈높이에 맞춘 친절한 언어로 풀어서 설명되었는지 확인한다.



In [ ]:
# [미션 09.] 정답 해설 - 모범 프롬프트 및 완성 산출물 리포트
mission_09_prompt = """
[신규 전입 신병 맞춤형 첫 주 온보딩 행정 문답집 및 용어 순화 대조표]
낯선 군 환경에 처음 배치된 신규 전입 신병이 일과 및 생활에 신속히 적응할 수 있도록, FAQ 데이터셋을 바탕으로 '전입 첫 주(D-Day ~ D+7) 시계열 행정 문답 5선'을 제작해 줘.
또한 신병들이 어려워하는 군 행정 전문 용어(온나라 상신, 인편 제출, 공람 확인, 제증명 발급 등)를 쉬운 일상어로 바꾼 [군 행정 용어 순화 대조표]를 완성해 줘.
"""

mission_09_result = """
## 1. 전입 첫 주 시계열 행정 문답 5선

### [문답 1] 전입 당일 (D-Day): 필수 서류 제출
- **신병 질문:** "오늘 막 자대에 전입 왔는데 행정반에 무슨 서류를 갖다 내야 하나요?"
- **챗봇 안내:**
  📌 **[질문 요약]**: 전입 당일 필수 행정 서류 제출 안내
  📝 **[단계별 절차]**:
     1단계: 행정반에서 신상기록부 서식 수령 후 작성
     2단계: 본인 명의 급여통장 사본과 가족관계증명서 1부 준비
     3단계: 오늘 일과 종료 전(17:00)까지 행정반 행정병에게 직접 전달
  📁 **[필요 서류]**: 가족관계증명서 1부, 급여통장 사본 1부, 신상명세서
  ⏰ **[유의사항]**: 오늘 내지 않으면 첫 달 월급 지급 계좌 등록이 지연될 수 있습니다.
  📞 **[담당 부서]**: 중대 행정반 (내선 101)
  🔍 **[근거]**: [근거: FAQ-F07]

---

## 2. 군 행정 용어 순화 대조표

| 군사 행정 전문 용어 | 순화된 쉬운 표현 | 순화 이유 및 기대효과 |
| :--- | :--- | :--- |
| 온나라 상신 | 컴퓨터로 담당 간부 승인 요청 | 군 행정망 결재 시스템 용어를 모르는 신병의 이해도 제고 |
| 인편 제출 | 행정반에 직접 방문하여 서류 전달 | 고어투 표현을 배제하고 구체적인 행동 방식으로 안내 |
| 공람 확인 | 공지사항 게시판 읽고 서명하기 | 무엇을 해야 하는지 명확한 행동 수칙으로 변환 |
| 제증명 발급 | 증명서(복무확인서 등) 떼기 | 일상 언어로 순화하여 위압감 해소 |
"""

# 렌더러 실행 (모범 해설 결과를 시각적으로 렌더링합니다)
show_result("미션 09. 신규 전입자용 문답 만들기", mission_09_prompt, mission_09_result)

